# Bluestock Fintech — Fund Performance Analytics
## Capstone Project I: Mutual Fund Analytics
**Author:** Surendhar | **Cohort:** 2025 | **Due:** 29 Jun 2026


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
print("Libraries loaded!")

## Load Data

In [ ]:
nav = pd.read_csv("data/nav_history.csv", parse_dates=["date"])
perf = pd.read_csv("data/scheme_performance.csv")
nifty50 = pd.read_csv("data/nifty50.csv", parse_dates=["date"])
nifty100 = pd.read_csv("data/nifty100.csv", parse_dates=["date"])
nav = nav.sort_values(["amfi_code","date"]).reset_index(drop=True)
nav = nav.merge(perf[["amfi_code","scheme_name","fund_house","category","expense_ratio_pct"]], on="amfi_code", how="left")
print(f"NAV: {nav.shape} | Schemes: {perf.shape[0]}")

## Task 1 — Daily Returns

In [ ]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
nav = nav.dropna(subset=["daily_return"])
print("Daily returns computed for all 40 schemes")
print(nav[["amfi_code","date","nav","daily_return"]].head(10))

## Task 2 — CAGR (1yr, 3yr, 5yr)

In [ ]:
def compute_cagr(group, years):
    end_date = group["date"].max()
    start_date = end_date - pd.DateOffset(years=years)
    subset = group[group["date"] >= start_date]
    if len(subset) < 2: return np.nan
    return (subset.iloc[-1]["nav"] / subset.iloc[0]["nav"]) ** (1/years) - 1

cagr_results = []
for code, group in nav.groupby("amfi_code"):
    cagr_results.append({
        "amfi_code": code,
        "scheme_name": group["scheme_name"].iloc[0],
        "cagr_1yr_pct": round(compute_cagr(group,1)*100,2),
        "cagr_3yr_pct": round(compute_cagr(group,3)*100,2),
        "cagr_5yr_pct": round(compute_cagr(group,5)*100,2),
    })
cagr_df = pd.DataFrame(cagr_results)
print(cagr_df.nlargest(10,"cagr_3yr_pct")[["scheme_name","cagr_1yr_pct","cagr_3yr_pct","cagr_5yr_pct"]].to_string(index=False))

## Task 3 — Sharpe Ratio

In [ ]:
RF = 0.065 / 252
sharpe_results = []
for code, group in nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    sharpe = ((returns - RF).mean() / returns.std()) * np.sqrt(252)
    sharpe_results.append({"amfi_code": code, "scheme_name": group["scheme_name"].iloc[0], "sharpe_ratio": round(sharpe,4)})
sharpe_df = pd.DataFrame(sharpe_results).sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
sharpe_df["sharpe_rank"] = range(1, len(sharpe_df)+1)
print(sharpe_df[["scheme_name","sharpe_ratio","sharpe_rank"]].to_string(index=False))

## Task 4 — Sortino Ratio

In [ ]:
sortino_results = []
for code, group in nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    downside_std = returns[returns < 0].std()
    sortino = ((returns - RF).mean() / downside_std) * np.sqrt(252) if downside_std > 0 else np.nan
    sortino_results.append({"amfi_code": code, "scheme_name": group["scheme_name"].iloc[0], "sortino_ratio": round(sortino,4)})
sortino_df = pd.DataFrame(sortino_results).sort_values("sortino_ratio", ascending=False)
print(sortino_df[["scheme_name","sortino_ratio"]].to_string(index=False))

## Task 5 — Alpha and Beta (OLS Regression)

In [ ]:
nifty100["return"] = nifty100["close"].pct_change()
alpha_beta_results = []
for code, group in nav.groupby("amfi_code"):
    fund_ret = group[["date","daily_return"]].dropna()
    merged = fund_ret.merge(nifty100[["date","return"]], on="date", how="inner")
    if len(merged) < 30: continue
    slope, intercept, r_value, _, _ = stats.linregress(merged["return"], merged["daily_return"])
    alpha_beta_results.append({"amfi_code": code, "scheme_name": group["scheme_name"].iloc[0],
        "alpha": round(intercept*252,4), "beta": round(slope,4), "r_squared": round(r_value**2,4)})
alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df.to_csv("alpha_beta.csv", index=False)
print(alpha_beta_df[["scheme_name","alpha","beta","r_squared"]].to_string(index=False))

## Task 6 — Maximum Drawdown

In [ ]:
drawdown_results = []
for code, group in nav.groupby("amfi_code"):
    group = group.sort_values("date")
    running_max = group["nav"].cummax()
    drawdown = group["nav"] / running_max - 1
    max_dd = drawdown.min()
    worst_idx = drawdown.idxmin()
    peak_idx = group["nav"][:worst_idx].idxmax()
    drawdown_results.append({"amfi_code": code, "scheme_name": group["scheme_name"].iloc[0],
        "max_drawdown_pct": round(max_dd*100,2),
        "peak_date": group.loc[peak_idx,"date"].date(),
        "trough_date": group.loc[worst_idx,"date"].date()})
drawdown_df = pd.DataFrame(drawdown_results)
print(drawdown_df.nsmallest(10,"max_drawdown_pct")[["scheme_name","max_drawdown_pct","peak_date","trough_date"]].to_string(index=False))

## Task 7 — Fund Scorecard (0-100)

In [ ]:
scorecard = cagr_df[["amfi_code","scheme_name","cagr_3yr_pct"]].copy()
scorecard = scorecard.merge(sharpe_df[["amfi_code","sharpe_ratio"]], on="amfi_code", how="left")
scorecard = scorecard.merge(alpha_beta_df[["amfi_code","alpha"]], on="amfi_code", how="left")
scorecard = scorecard.merge(perf[["amfi_code","expense_ratio_pct"]], on="amfi_code", how="left")
scorecard = scorecard.merge(drawdown_df[["amfi_code","max_drawdown_pct"]], on="amfi_code", how="left")
scorecard["return_rank"] = scorecard["cagr_3yr_pct"].rank(pct=True)*100
scorecard["sharpe_rank_score"] = scorecard["sharpe_ratio"].rank(pct=True)*100
scorecard["alpha_rank"] = scorecard["alpha"].rank(pct=True)*100
scorecard["expense_rank"] = (1-scorecard["expense_ratio_pct"].rank(pct=True))*100
scorecard["dd_rank"] = (1-scorecard["max_drawdown_pct"].rank(pct=True))*100
scorecard["fund_score"] = (0.30*scorecard["return_rank"]+0.25*scorecard["sharpe_rank_score"]+
    0.20*scorecard["alpha_rank"]+0.15*scorecard["expense_rank"]+0.10*scorecard["dd_rank"]).round(2)
scorecard = scorecard.sort_values("fund_score", ascending=False).reset_index(drop=True)
scorecard["overall_rank"] = range(1, len(scorecard)+1)
scorecard[["amfi_code","scheme_name","cagr_3yr_pct","sharpe_ratio","alpha",
    "expense_ratio_pct","max_drawdown_pct","fund_score","overall_rank"]].to_csv("fund_scorecard.csv", index=False)
print(scorecard[["scheme_name","fund_score","overall_rank"]].to_string(index=False))

## Task 8 — Benchmark Comparison Chart

In [ ]:
from IPython.display import Image
Image("charts/benchmark_comparison.png", width=900)